In [1]:
#Bibliotheken importieren:

import requests
from bs4 import BeautifulSoup as soup
import lxml.etree as ET
import unicodedata
import pandas as pd

# Datenaufbereitung

In [7]:
# Funktion mit Schleife: 

def dnb_sru(query, mdformat):
    
    base_url = "https://services.dnb.de/sru/dnb"
    params = {'recordSchema' : mdformat,
          'operation': 'searchRetrieve',
          'version': '1.1',
          'maximumRecords': '100',
          'query': query
         }
    
    i = 0 
    records = []

    while i % 100 == 0: 
        params.update({'startRecord': i})       
        r = requests.get(base_url, params=params)
        xml = soup(r.content, features="xml")
        result = xml.find_all('record', {'type':'Bibliographic'})
        records+=result
        num_results = len(result)
        if num_results != 100:
            break
        else: 
            i += 100  
        
    return records

In [8]:
result = dnb_sru("sw=Kreuzzüge", "MARC21-xml")
print(len(result))

706


In [10]:
print(result[1])

<record type="Bibliographic" xmlns="http://www.loc.gov/MARC21/slim">
<leader>00000nam a2200000uc 4500</leader>
<controlfield tag="001">1392999154</controlfield>
<controlfield tag="003">DE-101</controlfield>
<controlfield tag="005">20260409161043.0</controlfield>
<controlfield tag="007">cr||||||||||||</controlfield>
<controlfield tag="008">260315s2026    gw |||||o|||| 00||||ger  </controlfield>
<datafield ind1=" " ind2=" " tag="015">
<subfield code="a">26,O04</subfield>
<subfield code="2">dnb</subfield>
</datafield>
<datafield ind1="7" ind2=" " tag="016">
<subfield code="2">DE-101</subfield>
<subfield code="a">1392999154</subfield>
</datafield>
<datafield ind1=" " ind2=" " tag="020">
<subfield code="a">9783565324057</subfield>
<subfield code="9">978-3-565-32405-7</subfield>
</datafield>
<datafield ind1="3" ind2=" " tag="024">
<subfield code="a">9783565324057</subfield>
</datafield>
<datafield ind1="7" ind2=" " tag="024">
<subfield code="2">urn</subfield>
<subfield code="a">urn:nbn:de:10

### Speichern:

In [ ]:
# mit Hilfe einer Schleife wird jeder Datensatz nacheinander in die Datei "datadump.xml" geschrieben. An den Anfang und das Ende der Datei werden zudem die beiden Texte 
# hinter "f.write" geschrieben, die eine Art Klammer um die XML-Datensätze bilden und so dafür sorgen, einige Basisinformationen mitzugeben und das XML valide zu halten. 

with open ("datadump.xml", "w", encoding="utf-8") as f:
    f.write(f'''<?xml version="1.0" encoding="utf-8"?><searchRetrieveResponse xmlns="http://www.loc.gov/zing/srw/"> 
                <version>1.1</version><numberOfRecords>{str(len(records))}</numberOfRecords>
                <records><record><recordSchema>MARC21-xml</recordSchema><recordPacking>xml</recordPacking><recordData>''')
    for record in records:
        f.write(str(record))
    f.write(f'''</recordData></record></records><echoedSearchRetrieveRequest><version>1.1</version>
                <xQuery xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:nil="true"/>
                <recordSchema>MARC21-xml</recordSchema></echoedSearchRetrieveRequest>
                </searchRetrieveResponse>''')

In [ ]:
output = [parse_record(record) for record in result]
df = pd.DataFrame(output)
df

In [ ]:
df_dates = df.loc[df['date'] != "none"]
df_dates


In [ ]:
print(df_dates.date)#.to_string(index=False))

In [ ]:
#df_dates['date'] = df['date'].str.strip("[]")
df_dates['date'] = df_dates['date'].str.replace('\D', '', regex=True)
df_dates

In [ ]:
#print(df_dates.date)#.to_string(index=False))
df_dates.query("date.str.len() > 4")

In [ ]:
df_dates['date'] = df_dates['date'].str[:4]
df_dates

In [ ]:
#df_dates.query("date.str.len() > 4")
print(df_dates.date.to_string(index=False))

In [ ]:
df_dates.to_csv("BiblioCon.csv", encoding="utf-8")

## Advanced

In [ ]:
def parse_record(record):
    
    ns = {"marc":"http://www.loc.gov/MARC21/slim"}
    xml = ET.fromstring(unicodedata.normalize("NFC", str(record)))
    
    #idn
    idn = xml.xpath("marc:controlfield[@tag = '001']", namespaces=ns)
    try:
        idn = idn[0].text
    except:
        idn = 'fail'
    
    # title
    title = xml.xpath("marc:datafield[@tag = '245']/marc:subfield[@code = 'a']", namespaces=ns)
    try:
        title = title[0].text
    except:
        title = "unknown"
        
    
    # author
    author = xml.xpath("marc:datafield[@tag = '100']/marc:subfield[@code = 'a']", namespaces=ns)
    try:
        author = author[0].text
    except:
        author = "unknown"
       
       
 
    #date
    date = xml.xpath("marc:datafield[@tag = '264']/marc:subfield[@code = 'c']", namespaces=ns)
    date2 = xml.xpath("marc:datafield[@tag = '502']/marc:subfield[@code = 'd']", namespaces=ns)
    date3 = xml.xpath("marc:datafield[@tag = '502']/marc:subfield[@code = 'a']", namespaces=ns)
    date4 = xml.xpath("marc:datafield[@tag = '773']/marc:subfield[@code = 'g']", namespaces=ns)
    test = ""
    if date:
        date = date[0].text
    elif date2:
        date = date2[0].text
    elif date3:
        date = date3[0].text
    elif date4:
        for item in date4:
            test+= item.text + ", "
        date = test
    else:
        date = "none"
        
 
    
    meta_dict = {"idn":idn, "title":title, "author":author, "date":date}
    
    return meta_dict


In [ ]:
output = [parse_record(record) for record in result]
df = pd.DataFrame(output)
df

In [ ]:
df.query("date.str.len() > 4")